# Cell1 自检

自动断言 cache、AMP、num_workers、batch 三类别、NaN/Inf、真实幅值和系统内存。

In [1]:
from pathlib import Path
import importlib
import sys

RUN_DIR = Path.cwd() / 'experiments/cgan_v1/runs/2026-06-07_01_phase2a_full_newcache'
TRAIN_DIR = RUN_DIR / '02_train'
if str(TRAIN_DIR) not in sys.path:
    sys.path.insert(0, str(TRAIN_DIR))

import train
importlib.reload(train)

config = train.load_config()
cell1_report = train.cell1_self_check(config)
cell1_report

{'timestamp': '2026-06-07 15:25:35 +0800',
 'cache_dir': '/home/liujia/rf_training_cache/cgan_phase2a_64x32x32_random_full1500_fp16_cache_20260607',
 'use_amp': False,
 'num_workers': 0,
 'available_memory_gb': 52.929344177246094,
 'batch_categories': ['carotid', 'muscle', 'phantom'],
 'batch_stats': {'input': {'shape': [3, 1536, 64, 32, 32],
   'dtype': 'torch.float32',
   'max_abs': 10619.09375,
   'mean': -0.020570039749145508,
   'mean_abs': 261.2459716796875,
   'has_nan': False,
   'has_inf': False},
  'label': {'shape': [3, 2, 64, 32, 32],
   'dtype': 'torch.float32',
   'max_abs': 21726.177734375,
   'mean': -3.3417727947235107,
   'mean_abs': 2217.3935546875,
   'has_nan': False,
   'has_inf': False},
  'baseline': {'shape': [3, 2, 64, 32, 32],
   'dtype': 'torch.float32',
   'max_abs': 48332.2578125,
   'mean': -5.2621750831604,
   'mean_abs': 5183.76708984375,
   'has_nan': False,
   'has_inf': False}},
 'status': 'self_check_passed'}

# Cell2 配置

打印本次运行的唯一配置源摘要。

In [2]:
cell2_report = train.cell2_config(config)
cell2_report

{'run_name': '2026-06-07_01_phase2a_full_newcache',
 'cache_dir': '/home/liujia/rf_training_cache/cgan_phase2a_64x32x32_random_full1500_fp16_cache_20260607',
 'epochs': 50,
 'use_amp': False,
 'num_workers': 0,
 'loss': {'type': 'lsgan_struct_carrier',
  'lambda_adv': 1.0,
  'lambda_struct': 10.0,
  'lambda_carrier': 0.5,
  'lowpass': {'type': 'anisotropic_gaussian_3d',
   'sigma_voxels': {'z': 6.0, 'x': 3.0, 'y': 3.0},
   'truncate': 3.0}},
 'sampler': {'type': 'stratified_category_batch_sampler',
  'samples_per_category': 1,
  'effective_batch_size': 3,
  'shuffle': True,
  'seed': 20260607,
  'purpose': 'keep carotid/muscle/phantom present in each batch for BatchNorm3d'}}

# Cell3 数据

构建 train/val dataset、分层 batch sampler、固定 val 探针索引。

In [3]:
data_state = train.cell3_data(config)
data_state['summary']

{'train_samples': 1050, 'val_samples': 225}

# Cell4 模型

构建 TinyResidualRFNet(BatchNorm3d)、Envelope2DPatchDiscriminator、lowpass 和 optimizer。

In [4]:
model_state = train.cell4_models(config)
model_state['summary']

{'device': 'cuda',
 'G_params': 264738,
 'D_params': 662721,
 'G_first_norm': 'BatchNorm3d'}

# Cell5 训练循环

按 config 跑 50 epoch；每 epoch 写 stability.csv 和 speckle_check.csv，snapshot epoch 写三联探针图和 checkpoint。

In [5]:
train_result = train.cell5_train_loop(config, data_state, model_state)
train_result['result']

epoch=001 batch=0001 D_real=1.46514 D_fake=0.151855 G_adv_w=0.248689 G_struct_w=34660.9 G_carrier_w=2659.16 G_total=37320.4 pred_max_abs=76592.8


epoch=001 batch=0050 D_real=0.121952 D_fake=0.0930244 G_adv_w=1.27617 G_struct_w=33254.9 G_carrier_w=2210.54 G_total=35466.8 pred_max_abs=36923.1


epoch=001 batch=0100 D_real=0.0386905 D_fake=0.072816 G_adv_w=1.10905 G_struct_w=13260 G_carrier_w=950.325 G_total=14211.4 pred_max_abs=18961.6


epoch=001 batch=0150 D_real=0.0711416 D_fake=0.0328639 G_adv_w=0.763144 G_struct_w=20847 G_carrier_w=1506.04 G_total=22353.8 pred_max_abs=38972.2


epoch=001 batch=0200 D_real=0.0153009 D_fake=0.0320528 G_adv_w=0.93614 G_struct_w=78724.9 G_carrier_w=6394.47 G_total=85120.3 pred_max_abs=171672


...

epoch=050 batch=0250 D_real=0.000269902 D_fake=0.000186944 G_adv_w=0.991228 G_struct_w=87585 G_carrier_w=6383.35 G_total=93969.3 pred_max_abs=161186


epoch=050 batch=0300 D_real=0.000171843 D_fake=0.000128741 G_adv_w=0.9832 G_struct_w=31938.2 G_carrier_w=2833.38 G_total=34772.6 pred_max_abs=108437


epoch=050 batch=0350 D_real=0.000353623 D_fake=0.000209014 G_adv_w=0.994993 G_struct_w=43993.1 G_carrier_w=3208.47 G_total=47202.6 pred_max_abs=76207


EPOCH_SUMMARY epoch=050 batches=350 D_real=0.00143261 D_fake=0.00265306 D_real_score=0.730482 D_fake_score=0.500602 G_adv_w=0.999326 G_struct_w=43135.2 G_carrier_w=3055.39 G_total=46191.6 pred_max_abs=280180 peakGB=2.608 memGB=47.744


{'timestamp': '2026-06-07 22:47:12 +0800',
 'status': 'completed',
 'abort_reason': None,
 'epochs_completed': 50,
 'epochs_expected': 50,
 'elapsed_sec': 26496.735916614532,
 'last_stability': {'epoch': 50,
  'batches': 350,
  'seconds': 534.6486856937408,
  'peak_cuda_mem_gb': 2.6080875396728516,
  'pred_max_abs': 280180.1875,
  'has_nan': 0,
  'has_inf': 0,
  'available_memory_gb': 47.743743896484375,
  'D_real': 0.001432613685465185,
  'D_fake': 0.0026530648964814777,
  'D_real_score': 0.7304820035185132,
  'D_fake_score': 0.5006023549182075,
  'D_loss': 0.0020428392849031036,
  'G_adv_raw': 0.9993258147580283,
  'G_struct_raw': 4313.523649204799,
  'G_carrier_raw': 6110.774782366071,
  'G_adv_weighted': 0.9993258147580283,
  'G_struct_weighted': 43135.23649204799,
  'G_carrier_weighted': 3055.3873911830356,
  'G_total': 46191.62325753348},
 'checkpoints': ['02_train/checkpoints/checkpoint_epoch010.pth',
  '02_train/checkpoints/checkpoint_epoch020.pth',
  '02_train/checkpoints/chec

# Cell6 损失曲线

根据 stability.csv 生成/刷新 loss 曲线。

In [6]:
cell6_report = train.cell6_loss_curves(train_result['stability_rows'])
cell6_report

{'loss_curve': '02_train/figures/phase2a_loss_curves.png'}

# Cell7 清理

释放 lock 和 CUDA cache。

In [7]:
cell7_report = train.cell7_cleanup()
cell7_report

{'status': 'cleanup_done', 'timestamp': '2026-06-07 22:47:12 +0800'}

# Cell8 自动摘要

生成 phase2a_summary.md；只报健康事实，不做质量结论。

In [8]:
cell8_report = train.cell8_auto_summary(config)
cell8_report

{'summary_path': '02_train/phase2a_summary.md'}